# Week 6 (starter): Advanced Retrieval, Transform and Measure

Extends Week 5. The transformation shown here is pseudo-relevance feedback (PRF), which is fully local and needs no key; HyDE is guarded behind a key. Cells marked **TODO (you)** are yours. Dependency: `sentence-transformers`. For HyDE: `pip install openai` and a Gemini key.

In [1]:
# I completely reformatted my week 5 assignment because it was fairly disorganized, but I kept the same corpus documents so we could see the improvement.
import os, re, pathlib, requests, numpy as np, faiss
from collections import Counter
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
!pip install -q faiss-cpu

os.environ["HF_HOME"] = str((pathlib.Path(".") / ".hf_cache").resolve())
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

DOC_URLS = {
    'astar_32u4': 'https://www.pololu.com/docs/0J61/all',
    'astar_robot_controller': 'https://www.pololu.com/docs/0J66/all',
    'zumo_32u4': 'https://www.pololu.com/docs/0J63/all',
    'zumo_shield': 'https://www.pololu.com/docs/0J57',
    'three_pi_plus': 'https://www.pololu.com/docs/0J83/all',
    'zumo_chassis': 'https://www.pololu.com/docs/0J54/all',
    'qtr_sensors': 'https://www.pololu.com/docs/0J19/all',
    'mc33926_motor_shield': 'https://www.pololu.com/docs/0j55/all'
}

def load_doc(url):
    response = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]): tag.decompose()
    chunks = []
    for element in soup.select("h1, h2, h3, h4, p, li"):
        text = re.sub(r"\s+", " ", element.get_text(" ", strip=True)).strip()
        if text: chunks.append(text)
    return "\n\n".join(chunks)

DOCS = {}
for name, url in DOC_URLS.items():
    try:
        DOCS[name] = load_doc(url)
        print(f"{name:22s} -> {len(DOCS[name]):,} characters")
    except requests.RequestException as e:
        print(f"{name:22s} -> FAILED: {e}")

def chunk_fixed(text, size, overlap):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        if end >= len(text): break
        start += size - overlap
    return chunks

def chunk_paragraph(text):
    return [re.sub(r"\s+", " ", p).strip() for p in text.split("\n\n") if p.strip()]

configs = {
    "small_120_20": {name: chunk_fixed(text, 120, 20) for name, text in DOCS.items()},
    "large_320_40": {name: chunk_fixed(text, 320, 40) for name, text in DOCS.items()},
    "paragraph": {name: chunk_paragraph(text) for name, text in DOCS.items()}
}

prepared = {}
for config_name, documents in configs.items():
    all_chunks, metadata = [], []
    for document_name, chunks in documents.items():
        for chunk_number, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            metadata.append({"document": document_name, "chunk_number": chunk_number})
    prepared[config_name] = {"chunks": all_chunks, "metadata": metadata}

def build_index(chunks):
    embeddings = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

indices = {name: build_index(data["chunks"]) for name, data in prepared.items()}

def retrieve(config_name, query, k=3):
    data, index = prepared[config_name], indices[config_name]
    query_embedding = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, positions = index.search(query_embedding, k)
    return [
        {
            "score": float(score),
            "text": data["chunks"][position],
            "document": data["metadata"][position]["document"],
            "chunk_number": data["metadata"][position]["chunk_number"]
        }
        for score, position in zip(scores[0], positions[0]) if position >= 0
    ]



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

astar_32u4             -> 81,327 characters
astar_robot_controller -> 69,057 characters
zumo_32u4              -> 101,830 characters
zumo_shield            -> 1,216 characters
three_pi_plus          -> 91,048 characters
zumo_chassis           -> 8,518 characters
qtr_sensors            -> 26,947 characters
mc33926_motor_shield   -> 32,927 characters


Batches:   0%|          | 0/129 [00:00<?, ?it/s]

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

#Part 2 - Query definition

In [2]:
qa = [
    ("What microcontroller does the A-Star 32U4 use?", "astar_32u4", ("ATmega32U4",)),
    ("What type of motor driver is used by the Zumo 32U4?", "zumo_32u4", ("DRV8838",)),
    ("How are the motors controlled on the Zumo 32U4?", "zumo_32u4", ("PWM",)),
    ("What voltage range can the Zumo 32U4 operate from?", "zumo_32u4", ("2.7 V to 10 V",)),
    ("What is the purpose of the encoders on the Zumo 32U4?", "zumo_32u4", ("encoder",)),
    ("What does the MC33926 motor driver shield control?", "mc33926_motor_shield", ("DC motors",)),
    ("How does PWM control motor speed?", "zumo_32u4", ("PWM",)),
    ("What sensors are included on the Zumo 32U4?", "zumo_32u4", ("sensors",)),
    ("What is the purpose of the QTR reflectance sensors?", "qtr_sensors", ("reflectance",)),
    ("What is the Zumo Shield designed to work with?", "zumo_shield", ("Arduino",)),
    ("What kind of motors are used with the Zumo chassis?", "zumo_chassis", ("motors",)),
    ("What programming environment can be used with the A-Star 32U4?", "astar_32u4", ("Arduino IDE",))
]

norm = lambda s: re.sub(r"\s+", " ", s).lower()

labels = {
    name: [
        {i for i, (doc, text) in enumerate(zip(prepared[name]["metadata"], prepared[name]["chunks"]))
         if doc["document"] == document and any(norm(fact) in norm(text) for fact in facts)}
        for _, document, facts in qa
    ]
    for name in configs
}


#Part 3 - PRF transformation

In [3]:
STOP = set("a an the of to in on for and or is are how do does what so that with by as it why can from this these those".split())

def prf_expand(config_name, query, feedback_k=3, add=6):
    initial_results = retrieve(config_name, query, k=feedback_k)
    words = re.findall(r"[a-zA-Z]+", " ".join(r["text"] for r in initial_results).lower())
    query_words = set(re.findall(r"[a-zA-Z]+", query.lower()))
    candidates = [w for w in words if w not in STOP and len(w) > 3 and w not in query_words]
    additions = [w for w, _ in Counter(candidates).most_common(add)]
    return query + " " + " ".join(additions)

for question, _, _ in qa:
    print("Original:", question)
    print("Expanded:", prf_expand("paragraph", question))
    print()


Original: What microcontroller does the A-Star 32U4 use?
Expanded: What microcontroller does the A-Star 32U4 use? programming arduino

Original: What type of motor driver is used by the Zumo 32U4?
Expanded: What type of motor driver is used by the Zumo 32U4? drivers datasheets some components found available

Original: How are the motors controlled on the Zumo 32U4?
Expanded: How are the motors controlled on the Zumo 32U4? control motor drivers library provides functions

Original: What voltage range can the Zumo 32U4 operate from?
Expanded: What voltage range can the Zumo 32U4 operate from? battery users want experiment alternative power

Original: What is the purpose of the encoders on the Zumo 32U4?
Expanded: What is the purpose of the encoders on the Zumo 32U4? encoder each when interrupt drive motor

Original: What does the MC33926 motor driver shield control?
Expanded: What does the MC33926 motor driver shield control? schematic dual pins board datasheet lines

Original: How does

For this section I used pseudo-relevance feedback (PRF) to expand the original query based on the first few documents that were retrieved. The idea behind this is that the first results should contain additional terminology that is related to what the user is asking about, which can then give the retriever more information to work with when searching the rest of the corpus. For example, the query asking what microcontroller the A-Star 32U4 uses was expanded with terms like “programming” and “arduino.” Some of the expansions were more useful than others and some of them were not really natural sentences, however the purpose was to add relevant vocabulary rather than rewrite the question completely. Overall I think PRF worked reasonably well for this corpus because most of the added terms came from the same technical documentation that the original query was already searching through.

#Part 4 - ReRanking

In [4]:
def rerank(config_name, query, candidates, k=3):
    query_embedding = embedder.encode([query], normalize_embeddings=True)[0]
    candidate_embeddings = embedder.encode([r["text"] for r in candidates], normalize_embeddings=True)
    semantic_scores = candidate_embeddings @ query_embedding
    query_terms = {w for w in re.findall(r"[a-zA-Z]+", query.lower()) if w not in STOP}
    reranked = []

    for result, semantic_score in zip(candidates, semantic_scores):
        document_terms = {w for w in re.findall(r"[a-zA-Z]+", result["text"].lower()) if w not in STOP}
        overlap = len(query_terms & document_terms)
        final_score = 0.8 * float(semantic_score) + 0.2 * overlap
        result_copy = result.copy()
        result_copy["semantic_score"] = float(semantic_score)
        result_copy["term_overlap"] = overlap
        result_copy["rerank_score"] = final_score
        reranked.append(result_copy)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:k]


#Part 5 - Retrieval pipeline

In [5]:
BASELINE_CONFIG = "paragraph"

def retrieve_prf_reranked(config_name, query, candidate_k=10, final_k=3):
    expanded_query = prf_expand(config_name, query)
    candidates = retrieve(config_name, expanded_query, k=candidate_k)
    return rerank(config_name, query, candidates, k=final_k)


#Part 6 - Before/after measurements

In [6]:
K = 3

def is_relevant(result, facts):
    return any(norm(fact) in norm(result["text"]) for fact in facts)

def precision_at_k(results, facts, k=3):
    return sum(is_relevant(result, facts) for result in results[:k]) / k

evaluation = []

for question, document, facts in qa:
    baseline_results = retrieve(BASELINE_CONFIG, question, k=K)
    baseline_precision = precision_at_k(baseline_results, facts, K)

    improved_results = retrieve_prf_reranked(BASELINE_CONFIG, question, candidate_k=10, final_k=K)
    improved_precision = precision_at_k(improved_results, facts, K)

    delta = improved_precision - baseline_precision

    evaluation.append({
        "question": question,
        "baseline": baseline_precision,
        "improved": improved_precision,
        "delta": delta
    })

    print(f"{question}\n  baseline P@3: {baseline_precision:.2f}\n  Week 6 P@3:   {improved_precision:.2f}\n  delta:        {delta:+.2f}\n")


What microcontroller does the A-Star 32U4 use?
  baseline P@3: 0.00
  Week 6 P@3:   0.00
  delta:        +0.00

What type of motor driver is used by the Zumo 32U4?
  baseline P@3: 0.33
  Week 6 P@3:   0.33
  delta:        +0.00

How are the motors controlled on the Zumo 32U4?
  baseline P@3: 0.00
  Week 6 P@3:   0.00
  delta:        +0.00

What voltage range can the Zumo 32U4 operate from?
  baseline P@3: 0.33
  Week 6 P@3:   0.33
  delta:        +0.00

What is the purpose of the encoders on the Zumo 32U4?
  baseline P@3: 0.67
  Week 6 P@3:   1.00
  delta:        +0.33

What does the MC33926 motor driver shield control?
  baseline P@3: 0.00
  Week 6 P@3:   0.00
  delta:        +0.00

How does PWM control motor speed?
  baseline P@3: 1.00
  Week 6 P@3:   1.00
  delta:        +0.00

What sensors are included on the Zumo 32U4?
  baseline P@3: 0.33
  Week 6 P@3:   1.00
  delta:        +0.67

What is the purpose of the QTR reflectance sensors?
  baseline P@3: 1.00
  Week 6 P@3:   1.00
  del

After running the same 12 questions from Week 5 through the new PRF and reranking pipeline, I found that the retrieval results improved overall, although the improvement was not consistent across every question. Three of the twelve queries improved while the other nine stayed exactly the same, and there were no queries that got worse. The biggest improvement was for the question asking what sensors are included on the Zumo 32U4, which increased from 0.33 to 1.00 precision@3. The questions about the purpose of the encoders and the motors used with the Zumo chassis also improved. I think this shows that the added vocabulary from PRF was useful when the original query did not give the retriever enough specific terminology to find all of the relevant chunks. At the same time, several questions were already retrieving the relevant information, so adding more terms did not make a noticeable difference.

#Part 7 - Aggregating the results

In [7]:
baseline_mean = np.mean([r["baseline"] for r in evaluation])
improved_mean = np.mean([r["improved"] for r in evaluation])
overall_delta = improved_mean - baseline_mean

improved_count = sum(r["delta"] > 0 for r in evaluation)
regressed_count = sum(r["delta"] < 0 for r in evaluation)
same_count = sum(r["delta"] == 0 for r in evaluation)

print(f"Baseline mean P@3 : {baseline_mean:.3f}")
print(f"Week 6 mean P@3   : {improved_mean:.3f}")
print(f"Overall delta     : {overall_delta:+.3f}")
print(f"Improved queries  : {improved_count}")
print(f"Regressed queries : {regressed_count}")
print(f"Unchanged queries : {same_count}")


Baseline mean P@3 : 0.472
Week 6 mean P@3   : 0.583
Overall delta     : +0.111
Improved queries  : 3
Regressed queries : 0
Unchanged queries : 9


Overall, the Week 5 baseline had a mean precision@3 of 0.472, while the Week 6 PRF and reranking system had a mean precision@3 of 0.583. This is an overall improvement of 0.111, or about 11.1 percentage points. Out of the 12 queries, three improved, nine stayed the same, and none regressed. I think the improvement is useful even though it was not a massive change across the entire query set, because the purpose of the transformation is to improve retrieval where the original query has trouble finding the correct information. The fact that most queries stayed the same also makes sense because some of the original questions were already retrieving relevant chunks. In those cases there was not much room for PRF and reranking to improve the results.

#Part 8 - Query analysis

In [8]:
query_types = {
    "What microcontroller does the A-Star 32U4 use?": "component identification",
    "What type of motor driver is used by the Zumo 32U4?": "component identification",
    "How are the motors controlled on the Zumo 32U4?": "mechanism",
    "What voltage range can the Zumo 32U4 operate from?": "specification",
    "What is the purpose of the encoders on the Zumo 32U4?": "purpose",
    "What does the MC33926 motor driver shield control?": "function",
    "How does PWM control motor speed?": "mechanism",
    "What sensors are included on the Zumo 32U4?": "component identification",
    "What is the purpose of the QTR reflectance sensors?": "purpose",
    "What is the Zumo Shield designed to work with?": "compatibility",
    "What kind of motors are used with the Zumo chassis?": "component identification",
    "What programming environment can be used with the A-Star 32U4?": "compatibility"
}

for row in evaluation:
    row["type"] = query_types[row["question"]]

for query_type in sorted(set(r["type"] for r in evaluation)):
    rows = [r for r in evaluation if r["type"] == query_type]
    print(f"{query_type:25s} n={len(rows)} mean delta={np.mean([r['delta'] for r in rows]):+.3f}")


compatibility             n=2 mean delta=+0.000
component identification  n=4 mean delta=+0.250
function                  n=1 mean delta=+0.000
mechanism                 n=2 mean delta=+0.000
purpose                   n=2 mean delta=+0.167
specification             n=1 mean delta=+0.000


Looking at the results by query type, component identification questions benefited the most, with an average improvement of 0.250, followed by purpose questions with an average improvement of 0.167. The other categories, including compatibility, function, mechanism, and specification questions, did not change. I think this makes sense because component identification questions can sometimes use broad terms that appear in a lot of different parts of the technical documentation, so adding more vocabulary from the initial results can help narrow down what the query is actually looking for. Purpose questions also benefited from the additional terminology. However, I would not consider these results to prove that PRF is always better for these types of questions because the evaluation set is only 12 questions and some categories only have one or two questions.

#Part 9 - Failure analysis

In [10]:
regressions = [r for r in evaluation if r["delta"] < 0]
improvements = [r for r in evaluation if r["delta"] > 0]

if regressions:
    print("Regression found:")
    for r in regressions:
        print(f"- {r['question']}: {r['delta']:+.2f}")
else:
    print("No query regressed.")
    print("The PRF + reranking pipeline improved or preserved P@3 for all 12 queries.")
    print("\nQueries that improved:")
    for r in improvements:
        print(f"- {r['question']}: {r['delta']:+.2f}")


No query regressed.
The PRF + reranking pipeline improved or preserved P@3 for all 12 queries.

Queries that improved:
- What is the purpose of the encoders on the Zumo 32U4?: +0.33
- What sensors are included on the Zumo 32U4?: +0.67
- What kind of motors are used with the Zumo chassis?: +0.33


There were no queries that regressed after applying PRF and reranking. The pipeline either improved or preserved precision@3 for all 12 queries, with three queries showing an improvement. The biggest improvement was the question about what sensors are included on the Zumo 32U4, which increased by 0.67, while the questions about the purpose of the encoders and the type of motors used with the Zumo chassis both improved by 0.33. I think this shows that the added terms from the initial retrieval were useful for finding more specific information about components in the documentation. The other nine queries stayed the same, which means the transformation did not negatively affect their retrieval results. While PRF can potentially fail if the initial results contain irrelevant vocabulary, that did not happen with these 12 queries, so the transformation held across this evaluation set.